# Project 1 - Classifying Handwritten Digits

In this project you will build a real neural network (similar to the one discussed 
in Notebook 06) in PyTorch, and train it to recognize handwritten digits.

You are going to use the MNIST dataset. It consists of 70,000 little pictures of 
the digits $0$ through $9$, each one written by hand. Your job is to look at a 
picture and predict which digit it shows.

**Table of Contents**

- **Part A - The data (provided).** The cells below are already written for you.
  They download MNIST, explain what is inside it, and visualize it. Make sure you 
  understand the idea that *an image is just a grid of numbers*, because that is 
  what your network actually sees.
- **Part B - Your implementation (to be filled in).** This is your work. You will
  write the data loader, the model, the training loop, and the evaluation. Each
  task has a clear explanation of exactly what is needed and lines to be filled in.
- **Part C - Visualizing your results (provided helpers).** A few plotting
  helpers are written for you. Once Part B runs, call them to see how your
  network did.

## Part A - The data

### A.1  Downloading MNIST

The first time you run the cell below it downloads the MNIST dataset into a local
`data/` folder. It comes with two pieces:

- the **training set** (60,000 images): the examples your network learns from,
- the **test set** (10,000 images): held-out examples we only use at the end
  to measure how well the network does on digits it has never seen.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets

# download=True fetches the data the first time, then loads from disk after that
train_data = datasets.MNIST(root="data", train=True,  download=True)
test_data  = datasets.MNIST(root="data", train=False, download=True)

print(f"Training images: {len(train_data):,}")
print(f"Test images:     {len(test_data):,}")

### A.2  What is inside MNIST?

Each item in the dataset is a pair: an **image** and its **label**.

- The image is a $28 \times 28$ grayscale picture of a single handwritten digit.
- The label is the integer it represents, somewhere from $0$ to $9$.

Let us pull out the first training example and look at what type of thing it
actually is.

In [ ]:
image, label = train_data[0] # the first image and its label

print("image size   :", image.size, "(width x height)")
print("label        :", label)

# Turn the image into a plain grid of numbers so we can inspect it
pixels = np.array(image)
print("\nas a numpy array:")
print("  dtype        :", pixels.dtype, " -> each pixel is a number from 0 to 255")
print("  smallest val :", pixels.min(), " (darkest / black)")
print("  largest val  :", pixels.max(), "(brightest / white)")

### A.3  Looking at some digits

Before doing anything mathematical, let's visually inspect the data. Here are
the first 40 training images with their labels on top. Notice how 
real handwriting is messy. Digits vary in slant, shape, and clarity, 
and the network must handle these differences.

In [ ]:
fig, axes = plt.subplots(4, 10, figsize=(12, 5.2))
for ax, (img, lab) in zip(axes.ravel(), train_data):
    ax.imshow(img, cmap="gray")
    ax.set_title(str(lab), fontsize=10)
    ax.axis("off")
fig.suptitle("40 handwritten digits from the MNIST training set", fontsize=13)
plt.tight_layout()
plt.show()

### A.4  An image is just a grid of numbers

A picture *looks* like a picture to us, but to a computer there is no picture at
all, it's only numbers. A grayscale image is a grid of pixels, and each pixel is
one number measuring brightness:

- 0 means the pixel is completely black,
- 255 means the pixel is completely white,
- values in between are shades of gray.

An MNIST image is a $28 \times 28$ grid, so it is $28 \times 28 = 784$ numbers.
Let us visualize the first image as a grid of numbers, and see what they look like.

In [ ]:
image, label = train_data[0]
pixels = np.array(image) # 28 x 28 grid of integers in 0..255

fig, (ax_img, ax_num) = plt.subplots(1, 2, figsize=(14, 7))

# Left: the digit as a picture
ax_img.imshow(pixels, cmap="gray")
ax_img.set_title(f"What you see: the digit '{label}'", fontsize=13)
ax_img.axis("off")

# Right: the same digit as a 28x28 table of numbers
ax_num.imshow(pixels, cmap="gray")
for r in range(28):
    for c in range(28):
        v = pixels[r, c]
        ax_num.text(c, r, str(v), ha="center", va="center",
                    fontsize=4.2, color="black" if v > 110 else "white")
ax_num.set_title("What the network sees: 784 numbers arranged on a 28x28 grid",
                 fontsize=13)
ax_num.set_xticks([]); ax_num.set_yticks([])

plt.tight_layout()
plt.show()

The full grid is dense and hard to understand, so here is a zoomed-in view. We zoom into a $10 \times 10$
patch near the middle of the same digit. Where the pen pressed, the numbers are large
(bright), the empty background is all zeros (black).

In [ ]:
# zoom into the middle section of the same digit
patch = pixels[8:18, 8:18]

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(patch, cmap="gray", vmin=0, vmax=255)
for r in range(patch.shape[0]):
    for c in range(patch.shape[1]):
        v = patch[r, c]
        ax.text(c, r, str(v), ha="center", va="center",
                fontsize=11, color="black" if v > 110 else "white")
ax.set_title("A 10x10 close-up", fontsize=13)
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

When you build the model in Part B, the input layer will have 784
neurons, one for each number in this grid. We "flatten" the
$28 \times 28$ grid into a single row of 784 values and feed them in.

### A.5  How many of each digit?

One last check on the data: are the ten digits represented roughly equally, or is
the dataset *imbalanced*? A model trained on an imbalanced dataset can get lazy and just
favor the common classes. The bar chart below counts how many times each digit
appears in the training set. MNIST turns out to be fairly balanced, every digit
shows up a few thousand times.

In [ ]:
labels = np.array(train_data.targets) # all 60,000 training labels
counts = np.bincount(labels, minlength=10)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(range(10), counts, color="tab:blue")
for d, n in enumerate(counts):
    ax.text(d, n + 200, f"{n:,}", ha="center", fontsize=9)
ax.set_xticks(range(10))
ax.set_xlabel("digit")
ax.set_ylabel("number of training images")
ax.set_title("MNIST is roughly balanced across the ten digits")
ax.set_ylim(0, counts.max() * 1.12)
plt.tight_layout()
plt.show()

> **Note**: It's important to always analyze (and visualize) the data before any training. You want to understand what you are working with, and check for any issues that might affect your model's performance.

### A.6  Tensors and a validation set

Two last pieces of setup before Part B, both done for you here.

**1. Tensors.** A network cannot consume images directly, it needs the numbers in a
**PyTorch tensor**. A tensor is an arbitrary-dimensional (1D, 2D, 3D, ...) array of numbers, and it is
the basic data structure in PyTorch.

`transforms.ToTensor()` converts each image to a tensor and also **normalizes** its
pixels from the integer range $0..255$ to the floating-point range $0..1$ (networks 
usually train better on smaller inputs). With this normalization step, The brightest pixel 
is 1.0, the darkest pixel is 0.0, and the in-between shades are somewhere in between.

**2. A validation set.** As previously discussed, the test set is held out until the very end,
so we cannot use it for *hyperparameter tuning* (e.g., choosing the learning rate and weight 
decay coefficient). So we create a **validation set** out of the training data: keep
50,000 images for training and set aside 10,000 to check against as often as we
like. So the final dataset will have three splits:

- **training set** (50,000) - the network learns its weights from these;
- **validation set** (10,000) - held out from training; used for hyperparameter tuning 
(you will do this in B.5);
- **test set** (10,000) - used for model evaluation at the very end, for an honest
  final score.

The cell below handles these for you.

In [ ]:
import torch
from torch.utils.data import random_split
from torchvision import transforms

# Convert the images to tensors. ToTensor() also rescales pixels 0..255 -> 0..1.
train_split = datasets.MNIST(root="data", train=True,  transform=transforms.ToTensor())
test_split  = datasets.MNIST(root="data", train=False, transform=transforms.ToTensor())

# Hold out a validation set: 50,000 training images and 10,000 for validation.
# A fixed seed makes the split reproducible; the test set is left untouched.
split_generator = torch.Generator().manual_seed(0)
train_split, val_split = random_split(train_split, [50_000, 10_000],
                                      generator=split_generator)

print(f"training images  : {len(train_split):,}")
print(f"validation images: {len(val_split):,}")
print(f"test images      : {len(test_split):,}")

## Part B - Your implementation

From here on, you will write the main implementation yourself. Each task below
explains what the code should do and marks the places you need to complete with
`# TODO`. **Note that some may take more than one line of code.** Do not forget 
to run each cell before moving on.

First, run the below cell to import the necessary libraries for your implementation.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms

# Reproducibility: fixing the seed makes your runs repeatable
torch.manual_seed(0)

# Train on the GPU if available, otherwise the CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

### B.1  The data loader

In Part A (A.6) the images are already converted to tensors and split into
`train_split`, `val_split`, and `test_split`. All that is left is to serve them to
the network in small **batches**, which is what a `DataLoader` does.

**What is needed:**

1. Wrap `train_split`, `val_split`, and `test_split` each in a `DataLoader`.
   - Use a batch size of 32.
   - **Shuffle** the training loader (so each epoch sees the data in a different
     order). The validation and test loaders do not need shuffling.

In [ ]:
# B.1 - wrap each split in a DataLoader.
train_loader = ...  # TODO
val_loader = ...    # TODO
test_loader = ...   # TODO

# sanity check: look at the shape of one training batch
images, labels = next(iter(train_loader))
print("images:", images.shape)   # should print [32, 1, 28, 28]
print("labels:", labels.shape)   # should print [32]

### B.2  The model

Now build the network from the handout. It is a fully-connected network with 
the architecture:

| layer          | size |
|----------------|------|
| input          | 784  |
| hidden layer 1 | 32   |
| hidden layer 2 | 16   |
| output         | 10   |

**What is needed:**

1. Define a model using PyTorch's `nn.Module`.
2. Input to the model will have the shape `[batch, 1, 28, 28]`. The first thing the model must
   do is flatten it to `[batch, 784]` (hint: use `nn.Flatten`).
3. Then three `nn.Linear` layers of the sizes in the table, with a ReLU
   activation after each hidden layer. Do not put an activation after the
   final layer. The 10 output numbers should be the raw scores (one per digit).
   The model outputs should have the shape `[batch, 10]` (a score for each digit).
4. Create the model and move it to `device`. Print it to check the layer sizes.

In [ ]:
# B.2 - create the model.
class DigitClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = ...          # TODO
        self.hidden_layer_1 = ...   # TODO
        self.activation_fn_1 = ...  # TODO
        self.hidden_layer_2 = ...   # TODO
        self.activation_fn_2 = ...  # TODO
        self.output_layer = ...     # TODO

    def forward(self, x):
        """ x has shape [batch, 1, 28, 28] coming in. the output should have shape [batch, 10] """
        return ... # TODO pass input x through the model

model = ... # TODO create the model and move it to `device`
print(model)

n_params = sum(p.numel() for p in model.parameters())
print()
print(f"Total number of parameters (weights + biases): {n_params:,}")

# TODO: Do the algebra manually to verify the number of parameters in the model matches 
# what you expect.


### B.3  The training code

Training the network means showing it batches of images, measuring how wrong its
predictions are with a loss function, and nudging the weights to reduce that
loss with an **optimizer**, which runs the gradient descent algorithm automatically.

**What is needed:**

Write a function with the signature `train(lr, wd, n_epochs=2, seed=42)`, that:

1. builds a **fresh** `DigitClassifier` on `device`,
2. uses `nn.CrossEntropyLoss()` and **Adam optimizer** with the given learning rate 
   and weight decay (which implements the L2 regularization idea from Notebook 04);
3. loops for `n_epochs` epochs over `train_loader`, and for each batch applies the
   *training step*,
4. records the **average loss per epoch** in a list and returns `(model, train_losses)`.

In [ ]:
# B.3 - implement the training code.
def train(lr, wd, n_epochs=2, seed=42):
    """Build a fresh model and train it with the Adam optimizer.
    Returns (model, train_losses): the trained model and a list of 
    average losses per epoch."""
    
    torch.manual_seed(seed) # set seed for reproducibility
    model = ...     # TODO build a fresh model and move it to device, same as above
    loss_fn   = ... # TODO use cross-entropy loss
    optimizer = ... # TODO create Adam optimizer

    train_losses = [] # list to record the average loss per epoch
    for epoch in range(1, n_epochs + 1):
        running_loss = 0.0 # to accumulate the loss for this epoch
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device) # move the data to the same device as the model

            # TODO implement the training step
            # 1. clear last step's gradients (hint: zero_grad() on optimizer)
            # 2. forward pass to get the model's predictions for this batch
            # 3. compute the `loss` by comparing predictions to the true labels
            # 4. compute the gradients (hint: backward() on loss)
            # 5. update the weights (hint: step() on optimizer)

            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch}/{n_epochs} - average loss: {avg_loss:.4f}")
        
        train_losses.append(avg_loss) # average loss for this epoch

    return model, train_losses


### B.4  Choosing hyperparameters with grid search on the validation set

You will tune two hyperparameters we've learned about:

- the **learning rate** (`lr`) - how big a step the optimizer takes;
- the **weight decay coefficient** (`wd`) - the regularization strength that
discourages overfitting.

You will tune these hyperparameters using **grid search**. When multiple hyperparameters 
are involved, a straightforward approach is to choose a small set of candidate values for 
each one and then evaluate every possible combination, i.e., the full Cartesian product of 
the candidate sets. In this case, you will search over the following grid:

$$
\text{lr} \in \{1\times10^{-3},\, 3\times10^{-3},\, 1\times10^{-2}\}, \qquad
\text{wd} \in \{1\times10^{-4},\, 3\times10^{-4},\, 1\times10^{-3}\},
$$

We will train a separate model for each combination and evaluate each on the 
validation set, and finally select the best combination.

**What is needed:**

1. Write a helper `calculate_accuracy(model, loader)` that returns the percentage of correct
   predictions over a given split.
2. With `LR = [1e-3, 3e-3, 1e-2]` and `WD = [1e-4, 3e-4, 1e-3]`, loop over **every**
   `(lr, wd)` pair. For each, call the training function from B.3 (train for 2 epochs) and after training, 
   measure the model's accuracy on `val_loader`.
3. Keep the winning pair as `best_lr, best_wd` (and `best_key = (best_lr, best_wd)`).

In [ ]:
# B.4 - grid-search learning rate and weight decay on the validation set.

def calculate_accuracy(model, loader):
    """Percentage of correct predictions over a given split."""
    model.eval() # turn on evaluation mode
    correct = total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = ...  # TODO run the model on these images to get the predicted scores for each class.
            preds = ...    # TODO select the class with the maximum predicted score for each image in the batch.
                           # `preds` should have shape [batch], each item being the predicted digit (0..9) for that image.
            correct = correct + ... # TODO increment the number of correct predictions in this batch
            total = total + len(images) # add the number of images in this batch to total
    
    model.train() # back to training mode
    return (correct / total) * 100


# Grid search: try EVERY combination of these learning rates and weight decays.
LR = [1e-3, 3e-3, 1e-2]
WD = [1e-4, 3e-4, 1e-3]

results = {}
best_choice, best_acc = None, -1.0
for lr in LR:
    for wd in WD:
        print(f"Training with: lr = {lr:<6g}  wd = {wd:<6g}")

        candidate, _ = ...  # TODO train a model with this (lr, wd) combination
        val_acc = ...       # TODO compute the validation accuracy of this candidate model

        results[(lr, wd)] = val_acc
        print(f"Training completed. Validation accuracy = {val_acc:.2f}%\n")
        if val_acc > best_acc:
            best_choice, best_acc = (lr, wd), val_acc

best_lr, best_wd = best_choice
print(f"\nbest combination: lr = {best_lr}, wd = {best_wd}  "
      f"(validation accuracy {best_acc:.2f}%)")


### B.5  Train the final model

Finally, train the final model with the winning hyperparameters. You can use a longer
training duration for the final model. Do 5 epochs here instead of 2 (`n_epochs=5`).

**What is needed:**

1. Train a final model using `best_lr` and `best_wd`, for 5 epochs.

In [ ]:
# B.5 - train the final model with the winning hyperparameters.
model, train_losses = ... # TODO train the final model with best_lr, best_wd, n_epochs=5

### B.6  Evaluation on the test set

Now do the final evaluation on the **test** set to get the honest
estimate of how the final model does on digits it has never seen.

**What is needed:**

1. Use your `calculate_accuracy` helper to measure the final model's accuracy on `test_loader`.

In [ ]:
# B.6 - evaluate the final model on the test set
accuracy = ... # TODO compute the test accuracy of the final model.
print(f"test accuracy: {accuracy:.2f}%")

## Part C - Visualize your results

These plotting helpers are written for you. Just run the cells below after you complete Part B 
to see how your network did.

In [ ]:
def plot_training_loss(train_losses):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    epochs = range(1, len(train_losses) + 1)
    ax.plot(epochs, train_losses, marker="o", color="tab:purple")
    ax.set_xlabel("epoch")
    ax.set_ylabel("average training loss")
    ax.set_title("Training loss should fall as the network learns")
    ax.set_xticks(list(epochs))
    plt.tight_layout()
    plt.show()


def show_correct_predictions(model, loader, device="cpu", n=20):
    model.eval()
    right_imgs, right_pred = [], []
    with torch.no_grad():
        for images, labels in loader:
            preds = model(images.to(device)).argmax(dim=1).cpu()
            match = preds == labels
            for img, pred in zip(images[match], preds[match]):
                right_imgs.append(img)
                right_pred.append(pred.item())
            if len(right_imgs) >= n:
                break

    if not right_imgs:
        print("No correct predictions - the model got every image wrong!")
        return
    right_imgs, right_pred = right_imgs[:n], right_pred[:n]

    cols = min(10, len(right_imgs))
    rows = (len(right_imgs) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(1.3 * cols, 1.9 * rows),
                             squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, img, pred in zip(axes.ravel(), right_imgs, right_pred):
        ax.imshow(img.squeeze(), cmap="gray")
        ax.set_title(f"pred {pred}", color="tab:green", fontsize=10)
    fig.suptitle("A sample of the network's correct predictions "
                 "(green = correct)", fontsize=12)
    plt.tight_layout()
    plt.show()


def show_incorrect_predictions(model, loader, device="cpu", n=20):
    model.eval()
    wrong_imgs, wrong_true, wrong_pred = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            preds = model(images.to(device)).argmax(dim=1).cpu()
            mism = preds != labels
            for img, true, pred in zip(images[mism], labels[mism], preds[mism]):
                wrong_imgs.append(img)
                wrong_true.append(true.item())
                wrong_pred.append(pred.item())
            if len(wrong_imgs) >= n:
                break

    if not wrong_imgs:
        print("No incorrect predictions - the model got every image right!")
        return
    wrong_imgs, wrong_true, wrong_pred = wrong_imgs[:n], wrong_true[:n], wrong_pred[:n]

    cols = min(10, len(wrong_imgs))
    rows = (len(wrong_imgs) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(1.3 * cols, 1.9 * rows),
                             squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, img, true, pred in zip(axes.ravel(), wrong_imgs, wrong_true, wrong_pred):
        ax.imshow(img.squeeze(), cmap="gray")
        ax.text(0.48, 1.04, f"pred {pred},", color="tab:red", ha="right",
                va="bottom", fontsize=10, transform=ax.transAxes)
        ax.text(0.52, 1.04, f"true {true}", color="tab:green", ha="left",
                va="bottom", fontsize=10, transform=ax.transAxes)
    fig.suptitle("Where the network went wrong "
                 "(red = its guess, green = the truth)", fontsize=12)
    plt.tight_layout()
    plt.show()

print("OK")

In [ ]:
plot_training_loss(train_losses)

In [ ]:
show_correct_predictions(model, test_loader, device=device)

In [ ]:
show_incorrect_predictions(model, test_loader, device=device)